In [2]:
%load_ext autoreload
%autoreload 2

import sys
import torch

sys.path.append('.')
from src.utils import *
# from src.visualization import *
from src.engine import *
from src.dataset import *
from src.utils import *

In [ ]:
# set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_WORKERS = 0  # Windows/Jupyter에서는 0이 가장 안정적
print(f"Device: {device} | num_workers: {NUM_WORKERS}")

run_dirs = make_run_dir('./outputs')

In [ ]:
import os
import copy
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import fbeta_score, precision_score, recall_score

def Additional_mobilenet_experiments(
        bad_weight_list,
        seed,
        device,
        n_splits,
        num_epoch,
        batch_size,
        num_workers,
        early_stop_patience
):
    TRAIN_DIR   = './data/new_k-fold_data/train'
    TEST_DIR    = './data/new_k-fold_data/test'
    VALID_EXTS  = (".png", ".jpg", ".jpeg", ".bmp", ".webp")
    CLASS_NAMES = ["good", "type1", "type2", "type3", "type4", "type5"]

    # ── Step 1. 전체 데이터 풀링 + Train/Test Split ───────────────────
    all_paths, all_labels = [], []
    for root_dir in [TRAIN_DIR, TEST_DIR]:
        for class_name in CLASS_NAMES:
            class_dir = os.path.join(root_dir, class_name)
            if not os.path.isdir(class_dir):
                continue
            for fname in sorted(os.listdir(class_dir)):
                if fname.lower().endswith(VALID_EXTS):
                    all_paths.append(os.path.join(class_dir, fname))
                    all_labels.append(0 if class_name == "good" else 1)
    all_labels_np = np.array(all_labels)
    print(f"전체: {len(all_paths)}장 (good={int((all_labels_np==0).sum())}, bad={int(all_labels_np.sum())})")

    set_seed(seed)
    orig_train_paths, test_paths, orig_train_labels_np, test_labels_np = full_train_test_split(
        all_paths=all_paths, seed=seed
    )
    print(f"train_pool: {len(orig_train_paths)}장 (bad={int(orig_train_labels_np.sum())})")
    print(f"test_set  : {len(test_paths)}장 (bad={int(test_labels_np.sum())})")

    # ── Step 2. Weight Sweep + K-Fold CV ─────────────────────────────
    print(f"\n[Step 2] Weight Sweep + {n_splits}-Fold CV")
    fold_splits, full_train_detail_labels = custom_Stratified_K_Fold(orig_train_paths, n_splits=n_splits, seed=seed)

    weight_scores = {}

    for w in bad_weight_list:
        class_w   = torch.tensor([1.0, float(w)], dtype=torch.float32, device=device)
        criterion = nn.CrossEntropyLoss(weight=class_w)
        fold_f2s  = []

        for fold_idx, (train_idx_cv, val_idx_cv) in enumerate(fold_splits):
            set_seed(seed + fold_idx)

            tr_paths  = [orig_train_paths[i] for i in train_idx_cv]
            val_paths = [orig_train_paths[i] for i in val_idx_cv]

            train_ds_cv = ScrewDataset(tr_paths, orig_train_labels_np[train_idx_cv], transform=train_transform)
            val_ds_cv   = ScrewDataset(val_paths, orig_train_labels_np[val_idx_cv],  transform=val_transform)

            train_loader_cv = DataLoader(train_ds_cv, batch_size=batch_size, shuffle=True,
                                   num_workers=num_workers, pin_memory=(device.type == "cuda"))
            val_loader_cv   = DataLoader(val_ds_cv,   batch_size=batch_size, shuffle=False,
                                   num_workers=num_workers, pin_memory=(device.type == "cuda"))

            model     = build_model_bin('mobilenet_v2').to(device)
            optimizer = optim.Adam(model.parameters(), lr=1e-4)

            best_f2, no_improve = 0.0, 0

            for epoch in range(num_epoch):
                model.train()
                running_train_loss = 0.0
                for imgs, lbls in train_loader_cv:
                    imgs, lbls = imgs.to(device), lbls.to(device)
                    optimizer.zero_grad()
                    loss = criterion(model(imgs), lbls)
                    loss.backward()
                    optimizer.step()
                    running_train_loss += loss.item()
                avg_train_loss = running_train_loss / max(len(train_loader_cv), 1)

                model.eval()
                running_val_loss = 0.0
                y_true_val, y_pred_val = [], []
                with torch.no_grad():
                    for imgs, lbls in val_loader_cv:
                        imgs, lbls = imgs.to(device), lbls.to(device)
                        logits = model(imgs)
                        running_val_loss += criterion(logits, lbls).item()
                        y_true_val.extend(lbls.cpu().numpy())
                        y_pred_val.extend(logits.argmax(dim=1).cpu().numpy())
                avg_val_loss = running_val_loss / max(len(val_loader_cv), 1)

                val_f2 = fbeta_score(y_true_val, y_pred_val, beta=2, pos_label=1, zero_division=0)
                if val_f2 > best_f2:
                    best_f2, no_improve = val_f2, 0
                else:
                    no_improve += 1

                if epoch == 0 or (epoch + 1) % 10 == 0:
                    print(f"    Epoch: {epoch+1:02d} | "
                          f"Train loss {avg_train_loss:.4f} | "
                          f"Val loss: {avg_val_loss:.4f} | "
                          f"Val F2: {val_f2:.4f} | "
                          f"Best F2: {best_f2:.4f}")

                if early_stop_patience > 0 and no_improve >= early_stop_patience:
                    print(f"    -> Early stop at epoch {epoch + 1} "
                          f"(no improve for {early_stop_patience} epochs)")
                    break

            fold_f2s.append(best_f2)
            print(f"  w={w}, fold{fold_idx+1}: Val F2={best_f2:.4f}\n")

        mean_f2 = float(np.mean(fold_f2s))
        std_f2  = float(np.std(fold_f2s))
        weight_scores[w] = mean_f2

        print(f'  ===========================================================')
        print(f"  w={w} → mean F2={mean_f2:.4f} ± {std_f2:.4f}")
        print(f'  ===========================================================\n')

    best_weight = max(weight_scores, key=weight_scores.__getitem__)
    print(f"\n→ best_weight = {best_weight}")

    # ── Step 3. train_pool 전체 재학습 (best val F2 checkpoint 저장) ──
    print(f"\n[Step 3] Full Retrain  weight={best_weight}  epochs={num_epoch}")
    set_seed(seed)

    # 20% monitoring split — checkpoint 선택에만 사용, 하이퍼파라미터 튜닝 아님
    re_train_paths, re_val_paths, re_train_labels, re_val_labels = train_test_split(
        orig_train_paths, orig_train_labels_np,
        test_size=0.2, stratify=full_train_detail_labels, random_state=seed
    )
    re_train_ds = ScrewDataset(re_train_paths, np.array(re_train_labels), transform=train_transform)
    re_val_ds   = ScrewDataset(re_val_paths,   np.array(re_val_labels),   transform=val_transform)

    re_train_loader = DataLoader(re_train_ds, batch_size=batch_size, shuffle=True,
                             num_workers=num_workers, pin_memory=(device.type == "cuda"))
    re_val_loader   = DataLoader(re_val_ds,   batch_size=batch_size, shuffle=False,
                             num_workers=num_workers, pin_memory=(device.type == "cuda"))

    class_w   = torch.tensor([1.0, float(best_weight)], dtype=torch.float32, device=device)
    criterion = nn.CrossEntropyLoss(weight=class_w)
    re_model  = build_model_bin('mobilenet_v2').to(device)
    optimizer = optim.Adam(re_model.parameters(), lr=1e-4, weight_decay=1e-2)

    history = {"train_loss": [], "val_loss": [], "val_f2": []}
    best_val_f2     = 0.0
    best_state_dict = copy.deepcopy(re_model.state_dict())
    best_epoch      = 0

    for epoch in range(num_epoch):
        re_model.train()
        running_loss = 0.0
        for imgs, lbls in re_train_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            optimizer.zero_grad()
            loss = criterion(re_model(imgs), lbls)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        history["train_loss"].append(running_loss / len(re_train_loader))

        re_model.eval()
        val_loss = 0.0
        y_true_mon, y_pred_mon = [], []
        with torch.no_grad():
            for imgs, lbls in re_val_loader:
                imgs, lbls = imgs.to(device), lbls.to(device)
                logits = re_model(imgs)
                val_loss += criterion(logits, lbls).item()
                y_true_mon.extend(lbls.cpu().numpy())
                y_pred_mon.extend(logits.argmax(dim=1).cpu().numpy())
        history["val_loss"].append(val_loss / len(re_val_loader))

        mon_f2 = fbeta_score(y_true_mon, y_pred_mon, beta=2, pos_label=1, zero_division=0)
        history["val_f2"].append(mon_f2)

        if mon_f2 > best_val_f2:
            best_val_f2     = mon_f2
            best_state_dict = copy.deepcopy(re_model.state_dict())
            best_epoch      = epoch + 1

        if epoch == 0 or (epoch + 1) % 10 == 0:
            print(f"  Ep {epoch+1:02d} | Train {history['train_loss'][-1]:.4f} | "
                  f"Val {history['val_loss'][-1]:.4f} | Val F2 {mon_f2:.4f} | Best F2 {best_val_f2:.4f}")

    print(f"  → Best checkpoint: epoch {best_epoch}  (val F2={best_val_f2:.4f})")

    # ── Step 4. Test 평가 (best checkpoint 사용) ──────────────────────
    print(f"\n[Step 4] Test Evaluation  (checkpoint from epoch {best_epoch})")
    re_model.load_state_dict(best_state_dict)
    re_model.eval()

    re_test_ds     = ScrewDataset(test_paths, test_labels_np, transform=val_transform)
    re_test_loader = DataLoader(re_test_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)

    y_true_test, y_pred_test = [], []
    with torch.no_grad():
        for imgs, lbls in re_test_loader:
            y_pred_test.extend(re_model(imgs.to(device)).argmax(dim=1).cpu().numpy())
            y_true_test.extend(lbls.numpy())

    y_true_test = np.array(y_true_test)
    y_pred_test = np.array(y_pred_test)
    test_f2   = fbeta_score(y_true_test, y_pred_test, beta=2, pos_label=1, zero_division=0)
    test_rec  = recall_score(y_true_test, y_pred_test, pos_label=1, zero_division=0)
    test_prec = precision_score(y_true_test, y_pred_test, pos_label=1, zero_division=0)
    print(f"  Precision={test_prec:.4f} | Recall={test_rec:.4f} | F2={test_f2:.4f}")

    # ── Loss / Val F2 커브 ────────────────────────────────────────────
    ep_range = range(1, len(history["train_loss"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    axes[0].plot(ep_range, history["train_loss"], label="Train Loss")
    axes[0].plot(ep_range, history["val_loss"],   label="Val Loss (monitor)", linestyle="--")
    axes[0].axvline(best_epoch, color="red", linestyle=":", alpha=0.7, label=f"Best epoch {best_epoch}")
    axes[0].set_title(f"Loss Curve | Seed={seed}")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
    axes[0].legend(); axes[0].grid(True, alpha=0.3)

    axes[1].plot(ep_range, history["val_f2"], label="Val F2 (monitor)", color="green")
    axes[1].axvline(best_epoch, color="red", linestyle=":", alpha=0.7, label=f"Best epoch {best_epoch}")
    axes[1].set_title(f"Val F2 | best_weight={best_weight}\n"
                      f"Test F2={test_f2:.4f}  Recall={test_rec:.4f}  Precision={test_prec:.4f}")
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("F2")
    axes[1].legend(); axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    from sklearn.metrics import confusion_matrix
    import seaborn as sns

    cm = confusion_matrix(y_true_test, y_pred_test)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['good', 'bad'], yticklabels=['good', 'bad'],
                annot_kws={"size": 14, "weight": "bold"}, cbar=False)
    plt.title(f'MobileNet-V2 / Confusion Matrix / Seed = {seed}', fontsize=16, fontweight='bold', pad=15)
    plt.xlabel('Predicted Label', fontsize=12, fontweight='bold')
    plt.ylabel('True Label', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

    return {
        "seed":           seed,
        "best_weight":    best_weight,
        "best_epoch":     best_epoch,
        "test_f2":        test_f2,
        "test_recall":    test_rec,
        "test_precision": test_prec,
        "history":        history,
    }

In [ ]:
results = []
r = Additional_mobilenet_experiments(
        bad_weight_list   = [1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0],
        seed              = 42,
        device            = device,
        n_splits          = 5,
        num_epoch         = 40,
        batch_size        = 16,
        num_workers       = NUM_WORKERS,
        early_stop_patience = 10,
    )
results.append(r)

In [ ]:
r = Additional_mobilenet_experiments(
        bad_weight_list   = [1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0],
        seed              = 123,
        device            = device,
        n_splits          = 5,
        num_epoch         = 40,
        batch_size        = 16,
        num_workers       = NUM_WORKERS,
        early_stop_patience = 10,
    )
results.append(r)

In [ ]:
r = Additional_mobilenet_experiments(
        bad_weight_list   = [1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0],
        seed              = 456,
        device            = device,
        n_splits          = 5,
        num_epoch         = 40,
        batch_size        = 16,
        num_workers       = NUM_WORKERS,
        early_stop_patience = 10,
    )
results.append(r)

In [ ]:
import pandas as pd

df = pd.DataFrame(results)[["seed", "best_weight", "test_precision", "test_recall", "test_f2"]]
df.columns = ["Seed", "Best Weight", "Precision", "Recall", "F2"]
print("\n=== 최종 요약 ===")
display(df)
print(f"\nF2  Mean ± Std : {df['F2'].mean():.4f} ± {df['F2'].std():.4f}")
print(f"Rec Mean ± Std : {df['Recall'].mean():.4f} ± {df['Recall'].std():.4f}")

In [7]:
# # 1. 여기에 논문/실험에서 나온 숫자를 직접 입력하세요!
# TP = 19  # 불량을 불량으로 맞춘 개수 (True Positive)
# FP = 0   # 정상을 불량으로 오진한 개수 (False Positive)
# FN = 0   # 불량을 정상으로 놓친 개수 (False Negative - 미탐)
# TN = 31  # 정상을 정상으로 맞춘 개수 (True Negative)

# print(f"--- [입력된 Confusion Matrix] ---")
# print(f"TP(불량 적중): {TP}, FP(정상 오진): {FP}")
# print(f"FN(불량 미탐): {FN}, TN(정상 적중): {TN}\n")

# # 2. 성능 지표 계산 (0으로 나누는 에러 방지 포함)
# # Precision (정밀도): 불량이라고 예측한 것 중 진짜 불량인 비율
# precision = TP / (TP + FP) if (TP + FP) > 0 else 0

# # Recall (재현율): 실제 불량 중 모델이 맞춘 불량의 비율
# recall = TP / (TP + FN) if (TP + FN) > 0 else 0

# # F2-Score: Recall에 가중치(beta=2)를 두어 미탐을 중요하게 평가
# if (precision + recall) > 0:
#     f2_score = (5 * precision * recall) / ((4 * precision) + recall)
# else:
#     f2_score = 0

# # 3. 결과 출력
# print(f"--- [성능 평가 결과 (Bad 기준)] ---")
# print(f"Precision : {precision:.4f}")
# print(f"Recall    : {recall:.4f}")
# print(f"F2-Score  : {f2_score:.4f}")

def cal():
    FP = 0
    TN = 31
    for i in range(20):
        TP = 19-i
        FN = i

        precision = TP / (TP + FP) if (TP + FP) > 0 else 0
        recall = TP / (TP + FN) if (TP + FN) > 0 else 0

        if (precision + recall) > 0:
            f2_score = (5 * precision * recall) / ((4 * precision) + recall)
        else:
            f2_score = 0

        print(f'TP: {TP}, FN: {FN}, Recall: {recall:.4f}, F2-Score: {f2_score:.4f}')

cal()

TP: 19, FN: 0, Recall: 1.0000, F2-Score: 1.0000
TP: 18, FN: 1, Recall: 0.9474, F2-Score: 0.9574
TP: 17, FN: 2, Recall: 0.8947, F2-Score: 0.9140
TP: 16, FN: 3, Recall: 0.8421, F2-Score: 0.8696
TP: 15, FN: 4, Recall: 0.7895, F2-Score: 0.8242
TP: 14, FN: 5, Recall: 0.7368, F2-Score: 0.7778
TP: 13, FN: 6, Recall: 0.6842, F2-Score: 0.7303
TP: 12, FN: 7, Recall: 0.6316, F2-Score: 0.6818
TP: 11, FN: 8, Recall: 0.5789, F2-Score: 0.6322
TP: 10, FN: 9, Recall: 0.5263, F2-Score: 0.5814
TP: 9, FN: 10, Recall: 0.4737, F2-Score: 0.5294
TP: 8, FN: 11, Recall: 0.4211, F2-Score: 0.4762
TP: 7, FN: 12, Recall: 0.3684, F2-Score: 0.4217
TP: 6, FN: 13, Recall: 0.3158, F2-Score: 0.3659
TP: 5, FN: 14, Recall: 0.2632, F2-Score: 0.3086
TP: 4, FN: 15, Recall: 0.2105, F2-Score: 0.2500
TP: 3, FN: 16, Recall: 0.1579, F2-Score: 0.1899
TP: 2, FN: 17, Recall: 0.1053, F2-Score: 0.1282
TP: 1, FN: 18, Recall: 0.0526, F2-Score: 0.0649
TP: 0, FN: 19, Recall: 0.0000, F2-Score: 0.0000


In [1]:
import numpy as np

f2 = [
0.7653,
0.8854,
0.7843,
0.8065,
0.8500,
]

mean = np.mean(f2)
std = np.std(f2)

print(f'{mean:.4f}, {std:.4f}')

0.8183, 0.0438
